In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Load the clustered dataset from Phase 4
df = pd.read_parquet("../data/processed/postings_clustered.parquet")
print(f"Loaded: {len(df):,} postings")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSalary scope check:")
print(f"  Real salaries: {(~df['salary_is_predicted']).sum():,}")
print(f"  Predicted salaries: {df['salary_is_predicted'].sum():,}")
print(f"  Has cluster label: {df['cluster_name'].notna().sum():,}")

Loaded: 2,835 postings
Columns: ['salary_is_predicted', 'salary_min', 'title', 'created', 'id', 'salary_max', 'description', '_collected_at', 'longitude', 'contract_type', 'latitude', 'contract_time', 'category_label', 'company_name', 'location_full', 'location_area', 'category_tag', 'found_under_keywords', 'n_keywords_matched', 'title_length', 'description_length', 'description_word_count', 'salary_midpoint', 'text_for_embedding', 'umap_x', 'umap_y', 'cluster', 'cluster_name']

Salary scope check:
  Real salaries: 1,012
  Predicted salaries: 1,823
  Has cluster label: 2,835


In [2]:
# Confirm what's in salary_is_predicted
print(f"Column dtype: {df['salary_is_predicted'].dtype}")
print(f"\nValue counts:")
print(df['salary_is_predicted'].value_counts(dropna=False))
print(f"\nFirst 10 values: {df['salary_is_predicted'].head(10).tolist()}")

Column dtype: bool

Value counts:
salary_is_predicted
True     1823
False    1012
Name: count, dtype: int64

First 10 values: [True, False, True, False, False, False, True, False, False, True]


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import re

# Build a feature dataframe
feat = df.copy()

# --- Categorical features ---
# Cluster label (the role archetype we just identified)
feat["cluster_str"] = feat["cluster"].astype(str)

# Location - extract just the region (first segment) e.g. "London", "Scotland"
def extract_region(loc_area):
    if not isinstance(loc_area, (list, np.ndarray)):
        return "Unknown"
    if len(loc_area) >= 2:
        return loc_area[1]  # second element is typically the region
    return loc_area[0] if len(loc_area) > 0 else "Unknown"

feat["region"] = feat["location_area"].apply(extract_region)

# Top 15 regions only; everything else becomes "Other"
top_regions = feat["region"].value_counts().head(15).index.tolist()
feat["region_grouped"] = feat["region"].apply(lambda r: r if r in top_regions else "Other")

# Contract type and time (handle nulls)
feat["contract_type"] = feat["contract_type"].fillna("unknown")
feat["contract_time"] = feat["contract_time"].fillna("unknown")
feat["category_label"] = feat["category_label"].fillna("Unknown")

# --- Numeric features ---
feat["title_word_count"] = feat["title"].str.split().str.len()
feat["title_has_senior"] = feat["title"].str.lower().str.contains(
    r"\b(senior|lead|principal|head|director|manager|chief)\b", regex=True
).astype(int)
feat["title_has_junior"] = feat["title"].str.lower().str.contains(
    r"\b(junior|graduate|trainee|apprentice|entry)\b", regex=True
).astype(int)

print("Feature engineering summary:")
print(f"  Region distribution (top 5):")
print(feat["region_grouped"].value_counts().head(5).to_string())
print(f"\n  Senior-flagged postings: {feat['title_has_senior'].sum()}")
print(f"  Junior-flagged postings: {feat['title_has_junior'].sum()}")
print(f"\n  Sample features for first posting:")
print(f"    cluster_name: {feat.iloc[0]['cluster_name']}")
print(f"    region_grouped: {feat.iloc[0]['region_grouped']}")
print(f"    title_has_senior: {feat.iloc[0]['title_has_senior']}")
print(f"    title_has_junior: {feat.iloc[0]['title_has_junior']}")
print(f"    salary_midpoint: £{feat.iloc[0]['salary_midpoint']:,.0f}")

Feature engineering summary:
  Region distribution (top 5):
region_grouped
London                1227
UK                     463
North West England     219
South East England     159
South West England     139

  Senior-flagged postings: 1057
  Junior-flagged postings: 91

  Sample features for first posting:
    cluster_name: Data Analyst
    region_grouped: Northern Ireland
    title_has_senior: 0
    title_has_junior: 0
    salary_midpoint: £39,567


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.sparse import hstack, csr_matrix

# --- Categorical features → one-hot encoding ---
cat_features = ["cluster_str", "region_grouped", "contract_type", "contract_time", "category_label"]
encoder = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_cat = encoder.fit_transform(feat[cat_features])
print(f"Categorical features: {X_cat.shape[1]} columns")

# --- Text features → TF-IDF on title ---
title_vectorizer = TfidfVectorizer(
    max_features=200,           # top 200 most informative words
    ngram_range=(1, 2),         # include bigrams (e.g. "machine learning")
    stop_words="english",
    min_df=5                    # ignore words appearing in <5 postings
)
X_text = title_vectorizer.fit_transform(feat["title"].fillna(""))
print(f"Text features (TF-IDF): {X_text.shape[1]} columns")

# --- Numeric features → standard scaling ---
num_features = ["title_word_count", "title_has_senior", "title_has_junior",
                "description_length", "n_keywords_matched"]
scaler = StandardScaler()
X_num = scaler.fit_transform(feat[num_features].fillna(0))
X_num_sparse = csr_matrix(X_num)
print(f"Numeric features: {X_num.shape[1]} columns")

# --- Combine all features into one matrix ---
X = hstack([X_cat, X_text, X_num_sparse]).tocsr()
print(f"\nFinal feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features")

# --- Target variable ---
y = feat["salary_midpoint"].values
print(f"Target: salary_midpoint, range £{y.min():,.0f} to £{y.max():,.0f}")
print(f"Mean: £{y.mean():,.0f}, Median: £{np.median(y):,.0f}")

# Keep track of feature names for later interpretation
feature_names = (
    encoder.get_feature_names_out(cat_features).tolist() +
    title_vectorizer.get_feature_names_out().tolist() +
    num_features
)
print(f"\nTotal named features: {len(feature_names)}")

Categorical features: 62 columns
Text features (TF-IDF): 200 columns
Numeric features: 5 columns

Final feature matrix: 2,835 rows × 267 features
Target: salary_midpoint, range £nan to £nan
Mean: £nan, Median: £nan

Total named features: 267


In [5]:
# How many missing salary_midpoints?
print(f"Total rows: {len(feat):,}")
print(f"Missing salary_midpoint: {feat['salary_midpoint'].isna().sum():,}")
print(f"Has salary_midpoint: {feat['salary_midpoint'].notna().sum():,}")
print()

# Check the underlying salary columns too
print(f"Missing salary_min: {feat['salary_min'].isna().sum()}")
print(f"Missing salary_max: {feat['salary_max'].isna().sum()}")
print()

# Stats on the non-null subset
clean_salaries = feat["salary_midpoint"].dropna()
print(f"Salary stats (non-null only):")
print(f"  Min: £{clean_salaries.min():,.0f}")
print(f"  Max: £{clean_salaries.max():,.0f}")
print(f"  Mean: £{clean_salaries.mean():,.0f}")
print(f"  Median: £{clean_salaries.median():,.0f}")
print(f"  25th percentile: £{clean_salaries.quantile(0.25):,.0f}")
print(f"  75th percentile: £{clean_salaries.quantile(0.75):,.0f}")

Total rows: 2,835
Missing salary_midpoint: 3
Has salary_midpoint: 2,832

Missing salary_min: 1
Missing salary_max: 3

Salary stats (non-null only):
  Min: £1
  Max: £352,500
  Mean: £62,373
  Median: £57,487
  25th percentile: £45,287
  75th percentile: £71,261


In [6]:
# Investigate the salary extremes
print("=== LOW outliers (under £15K) ===")
low = feat[feat["salary_midpoint"] < 15000].sort_values("salary_midpoint")
print(f"Count: {len(low)}")
print(low[["title", "company_name", "salary_min", "salary_max", "salary_is_predicted"]].head(15).to_string())
print()

print("\n=== HIGH outliers (over £200K) ===")
high = feat[feat["salary_midpoint"] > 200000].sort_values("salary_midpoint", ascending=False)
print(f"Count: {len(high)}")
print(high[["title", "company_name", "salary_min", "salary_max", "salary_is_predicted"]].head(15).to_string())

=== LOW outliers (under £15K) ===
Count: 8
                                                           title                       company_name  salary_min  salary_max  salary_is_predicted
593                                      HR Data Privacy Analyst             Deloitte - Recruitment         1.0         1.0                False
1535                                       Software Engineer C++                     Sirius Digital        25.0        25.0                False
668             Market Data Administration & Data Vendor Analyst                           DonePlus        65.0        85.0                False
1151                                          Data Scientist III          Fanatics Betting & Gaming        99.0       124.0                False
814                                               HR Data Analst                     Insight Select       200.0       230.0                False
2452                                          Power BI developer  Pure Resourcing Solut

In [7]:
# Apply outlier filtering
before = len(feat)
n_dropped = 0

# Drop low outliers (< £15K - obvious data errors)
low_mask = feat["salary_midpoint"] < 15000
n_dropped += low_mask.sum()
feat = feat[~low_mask].copy()
print(f"Dropped {low_mask.sum()} low-salary outliers (<£15K)")

# Cap high outliers at £200K
high_mask = feat["salary_midpoint"] > 200000
n_capped = high_mask.sum()
feat.loc[feat["salary_midpoint"] > 200000, "salary_midpoint"] = 200000
print(f"Capped {n_capped} high-salary outliers at £200K")

# Also drop the 3 missing-salary rows
missing_mask = feat["salary_midpoint"].isna()
n_dropped += missing_mask.sum()
feat = feat[~missing_mask].copy()
print(f"Dropped {missing_mask.sum()} missing-salary rows")

print(f"\nClean modelling dataset: {len(feat):,} rows (was {before:,})")
print(f"\nFinal salary stats:")
print(f"  Min: £{feat['salary_midpoint'].min():,.0f}")
print(f"  Max: £{feat['salary_midpoint'].max():,.0f}")
print(f"  Mean: £{feat['salary_midpoint'].mean():,.0f}")
print(f"  Median: £{feat['salary_midpoint'].median():,.0f}")
print(f"\nBy salary type:")
print(f"  Real salaries: {(~feat['salary_is_predicted']).sum():,}")
print(f"  Predicted salaries: {feat['salary_is_predicted'].sum():,}")

Dropped 8 low-salary outliers (<£15K)
Capped 6 high-salary outliers at £200K
Dropped 3 missing-salary rows

Clean modelling dataset: 2,824 rows (was 2,835)

Final salary stats:
  Min: £15,090
  Max: £200,000
  Mean: £62,458
  Median: £57,500

By salary type:
  Real salaries: 1,001
  Predicted salaries: 1,823


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.sparse import hstack, csr_matrix

# === Re-build feature matrix on filtered data ===

# Categorical features → one-hot
cat_features = ["cluster_str", "region_grouped", "contract_type", "contract_time", "category_label"]
encoder = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_cat = encoder.fit_transform(feat[cat_features])

# Text features → TF-IDF
title_vectorizer = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=5
)
X_text = title_vectorizer.fit_transform(feat["title"].fillna(""))

# Numeric features → scaled
num_features = ["title_word_count", "title_has_senior", "title_has_junior",
                "description_length", "n_keywords_matched"]
scaler = StandardScaler()
X_num = scaler.fit_transform(feat[num_features].fillna(0))
X_num_sparse = csr_matrix(X_num)

# Combine
X = hstack([X_cat, X_text, X_num_sparse]).tocsr()
y = feat["salary_midpoint"].values

# Track which rows are real vs predicted (we'll split on this later)
is_predicted = feat["salary_is_predicted"].values

# Feature names for later interpretation
feature_names = (
    encoder.get_feature_names_out(cat_features).tolist() +
    title_vectorizer.get_feature_names_out().tolist() +
    num_features
)

print(f"Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target: {len(y):,} salary values")
print(f"  Real salaries: {(~is_predicted).sum():,}")
print(f"  Predicted salaries: {is_predicted.sum():,}")
print(f"  Range: £{y.min():,.0f} to £{y.max():,.0f}")

Feature matrix: 2,824 rows × 267 features
Target: 2,824 salary values
  Real salaries: 1,001
  Predicted salaries: 1,823
  Range: £15,090 to £200,000


In [9]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import make_scorer
import time

# 5-fold cross-validation (same fold structure for all models, for fair comparison)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Models to compare
models = {
    "Baseline (predict mean)": DummyRegressor(strategy="mean"),
    "Ridge Regression": Ridge(alpha=1.0, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, n_jobs=-1, random_state=42, verbosity=0)
}

def evaluate_models(X, y, label):
    print(f"\n{'=' * 60}")
    print(f"  TARGET: {label} (n={len(y):,})")
    print(f"{'=' * 60}")
    
    results = []
    for model_name, model in models.items():
        start = time.time()
        # R² (negative for sklearn's "scoring" convention; we'll flip back)
        r2_scores = cross_val_score(model, X, y, cv=cv, scoring="r2", n_jobs=-1)
        # MAE (negative MAE in sklearn convention)
        mae_scores = -cross_val_score(model, X, y, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1)
        elapsed = time.time() - start
        
        results.append({
            "model": model_name,
            "r2_mean": r2_scores.mean(),
            "r2_std": r2_scores.std(),
            "mae_mean": mae_scores.mean(),
            "mae_std": mae_scores.std(),
            "elapsed_s": elapsed
        })
        print(f"  {model_name:30s}  R²={r2_scores.mean():.3f}±{r2_scores.std():.3f}  MAE=£{mae_scores.mean():,.0f}±£{mae_scores.std():,.0f}  ({elapsed:.1f}s)")
    
    return pd.DataFrame(results)

# Real salary subset
real_mask = ~is_predicted
X_real = X[real_mask]
y_real = y[real_mask]
results_real = evaluate_models(X_real, y_real, "EMPLOYER-POSTED SALARIES")

# Predicted (Adzuna estimate) subset
pred_mask = is_predicted
X_pred = X[pred_mask]
y_pred = y[pred_mask]
results_pred = evaluate_models(X_pred, y_pred, "ADZUNA-PREDICTED SALARIES")

print(f"\n{'=' * 60}")
print("  COMPARISON")
print(f"{'=' * 60}")
print(f"\nReal salaries dataset: {(~is_predicted).sum():,} postings")
print(results_real.to_string(index=False))
print(f"\nPredicted salaries dataset: {is_predicted.sum():,} postings")
print(results_pred.to_string(index=False))


  TARGET: EMPLOYER-POSTED SALARIES (n=1,001)
  Baseline (predict mean)         R²=-0.011±0.011  MAE=£28,280±£1,996  (18.9s)
  Ridge Regression                R²=0.459±0.078  MAE=£19,502±£839  (8.4s)
  Random Forest                   R²=0.495±0.043  MAE=£17,123±£976  (22.3s)
  XGBoost                         R²=0.512±0.068  MAE=£16,925±£773  (7.9s)

  TARGET: ADZUNA-PREDICTED SALARIES (n=1,823)
  Baseline (predict mean)         R²=-0.008±0.004  MAE=£13,039±£667  (0.1s)
  Ridge Regression                R²=0.357±0.056  MAE=£10,044±£750  (0.2s)
  Random Forest                   R²=0.354±0.079  MAE=£10,018±£842  (38.2s)
  XGBoost                         R²=0.368±0.074  MAE=£9,730±£794  (6.7s)

  COMPARISON

Real salaries dataset: 1,001 postings
                  model   r2_mean   r2_std     mae_mean     mae_std  elapsed_s
Baseline (predict mean) -0.011359 0.011367 28279.693662 1995.952304  18.928027
       Ridge Regression  0.459330 0.078192 19501.683308  838.921945   8.359921
          R

In [10]:
# Save the model comparison results
results_real.to_csv("../data/processed/model_results_real.csv", index=False)
results_pred.to_csv("../data/processed/model_results_predicted.csv", index=False)
print("Saved:")
print(f"  ../data/processed/model_results_real.csv")
print(f"  ../data/processed/model_results_predicted.csv")

Saved:
  ../data/processed/model_results_real.csv
  ../data/processed/model_results_predicted.csv


In [11]:
from xgboost import XGBRegressor

# Train one XGBoost on each dataset to get feature importances
print("Training XGBoost on real salaries for feature importance...")
xgb_real = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, n_jobs=-1, random_state=42, verbosity=0)
xgb_real.fit(X[real_mask], y[real_mask])

print("Training XGBoost on Adzuna predictions for feature importance...")
xgb_pred = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, n_jobs=-1, random_state=42, verbosity=0)
xgb_pred.fit(X[pred_mask], y[pred_mask])

# Get feature importances
imp_real = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_real.feature_importances_
}).sort_values("importance", ascending=False)

imp_pred = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_pred.feature_importances_
}).sort_values("importance", ascending=False)

print("\n" + "=" * 70)
print("TOP 20 FEATURES — Real (employer-posted) salaries")
print("=" * 70)
print(imp_real.head(20).to_string(index=False))

print("\n" + "=" * 70)
print("TOP 20 FEATURES — Adzuna-predicted salaries")
print("=" * 70)
print(imp_pred.head(20).to_string(index=False))

# Save for the README
imp_real.to_csv("../data/processed/feature_importance_real.csv", index=False)
imp_pred.to_csv("../data/processed/feature_importance_predicted.csv", index=False)
print("\nSaved feature importance CSVs.")

Training XGBoost on real salaries for feature importance...
Training XGBoost on Adzuna predictions for feature importance...

TOP 20 FEATURES — Real (employer-posted) salaries
                            feature  importance
                   title_has_senior    0.056899
              region_grouped_London    0.045597
              contract_time_unknown    0.032327
                   title_has_junior    0.031603
                           software    0.027632
                              power    0.027460
                              staff    0.026568
category_label_Scientific & QA Jobs    0.025703
             contract_type_contract    0.025077
                            analyst    0.020423
                                 12    0.019445
                               head    0.019355
                     data architect    0.019104
                         management    0.017770
                             remote    0.016229
                           business    0.016187
        

In [12]:
# Quick summary for the README
print("\nSUMMARY OF PHASE 5 FINDINGS:\n")

print(f"Modelling comparison (5-fold CV, XGBoost):")
print(f"  Real salaries (n=1,001):     R² = 0.512, MAE = £16,925")
print(f"  Adzuna predictions (n=1,823): R² = 0.368, MAE = £9,730")
print(f"\nReal salaries are MORE predictable than Adzuna's predictions.")
print(f"Likely explanation: Adzuna's model uses signals not in our features.")
print(f"\nTop drivers of real salaries:")
print(f"  1. Senior flag in title (5.7%)")
print(f"  2. London region (4.6%)")
print(f"  3. Contract time unknown (3.2%)")
print(f"  4. Junior flag in title (3.2%)")
print(f"\nTop drivers of Adzuna's predictions:")
print(f"  1. Senior flag in title (7.2%)")
print(f"  2. Junior flag in title (4.5%)")
print(f"  3. 'analyst' in title (3.1%)")
print(f"  4. 'president' in title (2.8%)")
print(f"\nKey methodological finding:")
print(f"  London region is a top-5 feature for real salaries.")
print(f"  London is NOT in the top 20 features for Adzuna's predictions.")
print(f"  Adzuna's prediction model appears to underweight UK location.")


SUMMARY OF PHASE 5 FINDINGS:

Modelling comparison (5-fold CV, XGBoost):
  Real salaries (n=1,001):     R² = 0.512, MAE = £16,925
  Adzuna predictions (n=1,823): R² = 0.368, MAE = £9,730

Real salaries are MORE predictable than Adzuna's predictions.
Likely explanation: Adzuna's model uses signals not in our features.

Top drivers of real salaries:
  1. Senior flag in title (5.7%)
  2. London region (4.6%)
  3. Contract time unknown (3.2%)
  4. Junior flag in title (3.2%)

Top drivers of Adzuna's predictions:
  1. Senior flag in title (7.2%)
  2. Junior flag in title (4.5%)
  3. 'analyst' in title (3.1%)
  4. 'president' in title (2.8%)

Key methodological finding:
  London region is a top-5 feature for real salaries.
  London is NOT in the top 20 features for Adzuna's predictions.
  Adzuna's prediction model appears to underweight UK location.
